In [ ]:
consolidated = None
datanames = None
util = None
display_util = None
columnproc = None
userfriendly = None

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

import numpy as np
from tqdm.auto import tqdm
from IPython.display import display, Markdown

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
missing_final_vals = ["", "nan", "na", "none", "None", None]
missing_list = missing_final_vals + ["not tested", "unknown"]

# Userfriendly Dataset

Based on the configuration in `config/coltypes.csv` the columns are split in metadata, endpoint and feature data. Time series are processed in accordance with the configuration in the same file. For example some columns
will have their maxmimum value taken, while some will remain as time series.

In [ ]:
display(
    Markdown(
        "Besides the empty values, these values are also considered missing: "
        + ", ".join([f"`{val}`" for val in missing_list])
    )
)

In [ ]:
consolidated = pd.read_parquet(consolidated)
display(Markdown("### Consolidated Data Loaded"))
display(
    Markdown(
        f"- **Shape**: {consolidated.shape[0]} rows × {consolidated.shape[1]} columns"
    )
)
display(
    Markdown(
        f"- **Datasets**: {consolidated.columns.get_level_values('dataset').nunique()}"
    )
)
display(
    Markdown(
        f"- **Memory usage**: {consolidated.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
    )
)

strs = consolidated.dtypes.apply(pd.api.types.is_string_dtype)
timeseries = consolidated.loc[:, strs].apply(
    lambda col: col.str.contains(";;", regex=False).any()
)


def build_anno_help(df, timeseries):
    tsgroups = (
        timeseries[timeseries]
        .reset_index("column")
        .groupby(["dataset"])
        .apply(lambda df: df["column"].to_list())
    )
    for k, v in list(tsgroups.to_dict().items()):
        selector = [(k, val) for val in v]
        rowtslen = (
            df.loc[:, selector].apply(lambda col: col.str.count(";;")).nunique(axis=1)
        )
        assert ((rowtslen == 1) | (rowtslen == 0)).all()
    toanno = df.columns.to_frame()
    toanno["timeseries"] = toanno.index.isin(timeseries[timeseries].index)
    toanno["coltype"] = "TODO"
    toanno["timeseries_agg"] = "TODO"
    toanno["timeseries_agg"] = toanno["timeseries_agg"].where(toanno["timeseries"], "")
    return toanno.reset_index(drop=True)

In [ ]:
coltypes = pd.read_csv(columnproc, sep=";").set_index(["dataset", "column"])
# coltypes.drop(columns=coltypes.columns[[0, 4]], inplace=True)
display(Markdown("### Column Type Configuration"))
display(Markdown(f"- **Total configured columns**: {len(coltypes)}"))
display(Markdown(f"- **Timeseries columns in config**: {coltypes['timeseries'].sum()}"))

In [ ]:
assert (
    coltypes.index.isin(consolidated.columns).all()
    and consolidated.columns.isin(coltypes.index).all()
), "Missing columns!"

with pd.option_context("future.no_silent_downcasting", True):
    mismatches = (
        pd.merge(
            timeseries.rename("Time Series seen"),
            coltypes["timeseries"].rename("Config Given"),
            how="right",
            left_index=True,
            right_index=True,
        )
        .infer_objects(copy=False)
        .fillna(False)
        .astype("bool")
    )
mismatches = mismatches[mismatches.sum(axis=1) == 1]

if mismatches.shape[0] > 0:
    display(mismatches)
    assert False, "Mismatching config!"


def fun_latest(elements):
    elements = elements.dropna()
    if elements.size == 0:
        return None
    else:
        return elements.iloc[-1]


def fun_keep(elements):
    return ";;".join(elements.astype("str"))


def fun_max(elements):
    return elements.max()


def fun_min(elements):
    return elements.min()


def fun_range(elements):
    return elements.max() - elements.min()


def fun_count(elements):
    return elements.size


funs = {
    "latest": fun_latest,
    "keep": fun_keep,
    "max": fun_max,
    "range": fun_range,
    "count": fun_count,
    "min": fun_min,
}

In [ ]:
resfile = Path(userfriendly)
res = []
display(Markdown("### Processing Columns by Type"))
for (dataset, istimeseries), i in tqdm(
    coltypes.groupby(["dataset", "timeseries"]).indices.items()
):
    cols = coltypes.iloc[i]
    extracted = consolidated.loc[:, cols.index].copy()
    newidx = pd.MultiIndex.from_frame(
        cols.reset_index().loc[:, list(cols.index.names) + ["coltype"]]
    )
    newidx.set_names(["dataset", "column", "coltype"], inplace=True)
    extracted = extracted.set_axis(newidx, axis=1)
    if not istimeseries:
        res.append(extracted.replace(missing_final_vals, np.nan))
    else:
        replacement = (
            (extracted.apply(lambda col: col.str.count(";;")).max(axis=1) + 1)
            .fillna(0)
            .astype("int")
            .apply(lambda count: ";;".join(("" for _ in range(count))))
        )
        namask = extracted.isna()
        longform = extracted.copy()
        for col in longform.columns:
            longform.loc[namask[col], col] = replacement[namask[col]]
        longform = (
            longform.apply(lambda col: col.str.split(";;"))
            .explode(list(extracted.columns))
            .replace(missing_list, np.nan)
        )
        for col in longform.columns:
            try:
                longform[col] = pd.to_numeric(longform[col])
            except ValueError:
                pass
        funstoapply = {
            extracted.columns[i]: funs[v]
            for i, v in enumerate(cols["timeseries_agg"].to_dict().values())
        }
        rename_tuples = [
            (
                extracted.columns[i][0],
                extracted.columns[i][1] + "__timeseries_" + v,
                extracted.columns[i][2],
            )
            for i, v in enumerate(cols["timeseries_agg"].to_dict().values())
        ]
        extracted = longform.groupby(longform.index.names, dropna=False).agg(
            funstoapply
        )
        extracted = extracted.set_axis(pd.MultiIndex.from_tuples(rename_tuples), axis=1)
        extracted.columns.set_names(["dataset", "column", "coltype"], inplace=True)
        res.append(extracted)
features = pd.concat(res, axis=1)

display(Markdown("### Processing Complete"))
display(
    Markdown(
        f"- **Final shape**: {features.shape[0]} rows × {features.shape[1]} columns"
    )
)
display(
    Markdown(
        f"- **Null values**: {features.isna().sum().sum()} ({100*features.isna().sum().sum()/(features.shape[0]*features.shape[1]):.1f}%)"
    )
)
display(Markdown("- **Data types**:"))

In [ ]:
features = features.replace(missing_final_vals, np.nan).copy()

In [ ]:
for col in features.columns:
    try:
        features[col] = pd.to_numeric(features[col])
    except ValueError:
        pass
features = features.infer_objects()

display(Markdown("### Data Types Inferred"))
display(
    Markdown(
        f"- **Numeric columns**: {features.select_dtypes(include=['number']).shape[1]}"
    )
)
display(
    Markdown(
        f"- **String/Object columns**: {features.select_dtypes(include=['object', 'string']).shape[1]}"
    )
)

features.to_parquet(resfile)